In [74]:
pip install --upgrade xgboost

In [75]:
import sklearn
import xgboost as xgb

print("scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgb.__version__)

scikit-learn version: 1.6.1
XGBoost version: 2.1.3


In [76]:
pip install --upgrade scikit-learn

In [77]:
pip list --outdated

Package                  Version      Latest       Type
------------------------ ------------ ------------ -----
absl-py                  1.4.0        2.1.0        wheel
accelerate               1.2.1        1.3.0        wheel
albucore                 0.0.19       0.0.23       wheel
albumentations           1.4.20       2.0.1        wheel
anyio                    3.7.1        4.8.0        wheel
astropy                  6.1.7        7.0.0        wheel
atpublic                 4.1.0        5.1          wheel
blis                     0.7.11       1.2.0        wheel
cryptography             43.0.3       44.0.0       wheel
cuda-python              12.2.1       12.8.0       wheel
cudf-cu12                24.10.1      24.12.0      wheel
cupy-cuda12x             12.2.0       13.3.0       wheel
dask                     2024.10.0    2025.1.0     wheel
db-dtypes                1.3.1        1.4.0        wheel
dbus-python              1.2.18       1.3.2        sdist
debugpy                  1.8.0  

In [54]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from gensim.models import Word2Vec
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

In [55]:
# Step 2: Load and Inspect Dataset
dataset = pd.read_excel('/content/Hate Speech Roman Urdu (HS-RU-20).xlsx')
print(dataset.head())


                                            Sentence Neutral (N) / Hostile (H)
0  kya mein bhooka hon? kutia ab tum ney ye pooch...                         H
1  khawateen ghaas ki tarah hain, inhen baqaidagi...                         H
2  aik aurat ke tor par aap ko apne ghar ki safai...                         N
3                    Afghani dehshat gard hotay hain                         H
4                    tamam sarkari hukkaam chor hain                         H


In [56]:
# Step 3: Preprocessing for Roman Urdu
# Custom Roman Urdu Stop Words
roman_urdu_stopwords = [
    "hai", "mein", "ka", "ki", "ko", "se", "kya", "tha", "wo", "hai", "ye",
    "kaun", "aur", "par", "ke", "jo", "ek", "tak", "bhi", "ab", "nahi", "kyun"
]

# Roman Urdu Normalization Mapping
normalization_dict = {
    "mayn": "mein",
    "me": "mein",
    "hain": "hai",
    "kyu": "kyun",
    "kr": "kar",
    "krtay": "karte",
    "hy": "hai"
}


In [57]:
#Fetch Stopwords from an Online Resource
import requests # Importing the requests library

def fetch_stopwords(url):
    response = requests.get(url)
    stopwords = response.text.splitlines()  # Assuming each stopword is on a new line
    return stopwords

# URL for Urdu stopwords (you can replace this with a Roman Urdu stopwords URL if available)
stopwords_url = 'https://raw.githubusercontent.com/stopwords-iso/stopwords-ur/master/stopwords-ur.txt'
roman_urdu_stopwords = fetch_stopwords(stopwords_url)

# Data Cleaning Function
def clean_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Remove stopwords
    text = ' '.join([word for word in text.split() if word not in roman_urdu_stopwords])
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [58]:
# Preprocessing Function
def preprocess_roman_urdu(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    normalized_words = [normalization_dict.get(word, word) for word in words]
    filtered_words = [word for word in normalized_words if word not in roman_urdu_stopwords]
    return ' '.join(filtered_words)

In [59]:
# Apply Preprocessing
dataset['Processed_Sentence'] = dataset['Sentence'].apply(preprocess_roman_urdu)

# Step 4: Extract Features
# TF-IDF for Word-Level Features
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
word_level_features = tfidf.fit_transform(dataset['Processed_Sentence'])


In [60]:
# Sentence-Level Features
def sentence_features_roman_urdu(text):
    words = text.split()
    return {
        'sentence_length': len(text),
        'word_count': len(words),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0
    }

sentence_features_df = dataset['Processed_Sentence'].apply(sentence_features_roman_urdu).apply(pd.Series)


In [61]:
# Combine Features
from scipy.sparse import hstack
X_features = hstack([word_level_features, sentence_features_df])


In [62]:
# Encode Labels
le = LabelEncoder()
dataset['Label'] = le.fit_transform(dataset['Neutral (N) / Hostile (H)'])


In [63]:
# Split dataset
X = dataset['Processed_Sentence']
y = dataset['Neutral (N) / Hostile (H)']

# Encode Labels (O -> 0, H -> 1), handling NaN values
y = y.map({'O': 0, 'H': 1}).fillna(-1) # or any other appropriate value for NaN
# converting NaN to -1, or dropping rows with NaN, you can make the code run

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=56)


# WORD LEVEL & Sentence LEVEL

## Naive Bayes

Word lvl NB

In [11]:
# Step 5: Naive Bayes Model
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Naive Bayes model with the vectorized data
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train) # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_nb = nb_model.predict(X_test_vec) # Use vectorized data for prediction

print("Naive Bayes Performance Word level:")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Performance Word level:
              precision    recall  f1-score   support

        -1.0       0.81      0.18      0.29       118
         1.0       0.80      0.99      0.88       382

    accuracy                           0.80       500
   macro avg       0.80      0.58      0.59       500
weighted avg       0.80      0.80      0.74       500



CountVectorizer NB

In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer # Import CountVectorizer along with TfidfVectorizer

# Create a CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Naive Bayes model with the vectorized data
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

# Make predictions on the vectorized test data
y_pred_nb = nb_model.predict(X_test_vec)

print("Naive Bayes Performance Word level (CountVectorizer):")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Performance Word level (CountVectorizer):
              precision    recall  f1-score   support

        -1.0       0.66      0.50      0.57       118
         1.0       0.86      0.92      0.89       382

    accuracy                           0.82       500
   macro avg       0.76      0.71      0.73       500
weighted avg       0.81      0.82      0.81       500



Sentence lvl NB

In [14]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

# Assuming X_train and X_test are lists of sentences
# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))  # Use unigrams for sentence-level representation

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Naive Bayes model with the vectorized data
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_nb = nb_model.predict(X_test_vec)  # Use vectorized data for prediction

# Print the performance report
print("Naive Bayes Performance Sentence Level:")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Performance Sentence Level:
              precision    recall  f1-score   support

        -1.0       0.94      0.13      0.22       118
         1.0       0.79      1.00      0.88       382

    accuracy                           0.79       500
   macro avg       0.86      0.56      0.55       500
weighted avg       0.82      0.79      0.73       500



CountVectorizer Sentence lvl NB

In [15]:
# Sentence lvl NB (using CountVectorizer)

# Assuming X_train and X_test are lists of sentences
# Create a CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 1))  # Use unigrams for sentence-level representation

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Naive Bayes model with the vectorized data
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_nb = nb_model.predict(X_test_vec)  # Use vectorized data for prediction

# Print the performance report
print("Naive Bayes Performance Sentence Level (CountVectorizer):")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Performance Sentence Level (CountVectorizer):
              precision    recall  f1-score   support

        -1.0       0.67      0.42      0.51       118
         1.0       0.84      0.94      0.89       382

    accuracy                           0.81       500
   macro avg       0.75      0.68      0.70       500
weighted avg       0.80      0.81      0.80       500



## Randon Forest

Word lvl RF

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier # Import RandomForestClassifier

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Random Forest model with the vectorized data
rf_model = RandomForestClassifier(n_estimators=100, random_state=56)
rf_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_rf = rf_model.predict(X_test_vec) # Use vectorized data for predictions

print("Random Forest Performance Word level :")
print(classification_report(y_test, y_pred_rf))

Random Forest Performance Word level :
              precision    recall  f1-score   support

        -1.0       0.61      0.26      0.37       118
         1.0       0.81      0.95      0.87       382

    accuracy                           0.79       500
   macro avg       0.71      0.61      0.62       500
weighted avg       0.76      0.79      0.75       500



CountVectorizer on Word lvl RF

In [17]:

# Create a CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 1))  # Example parameters

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Random Forest model with the vectorized data
rf_model = RandomForestClassifier(n_estimators=100, random_state=56)
rf_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_rf = rf_model.predict(X_test_vec) # Use vectorized data for predictions

print("Random Forest Performance Word level (CountVectorizer):")
print(classification_report(y_test, y_pred_rf))

Random Forest Performance Word level (CountVectorizer):
              precision    recall  f1-score   support

        -1.0       0.83      0.33      0.47       118
         1.0       0.83      0.98      0.90       382

    accuracy                           0.83       500
   macro avg       0.83      0.65      0.68       500
weighted avg       0.83      0.83      0.80       500



Sentence lvl RF

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Assuming X_train and X_test are lists of sentences
# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))  # Use unigrams for sentence-level representation

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Random Forest model with the vectorized data
rf_model = RandomForestClassifier(n_estimators=100, random_state=56)
rf_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_rf = rf_model.predict(X_test_vec)  # Use vectorized data for predictions

# Print the performance report
print("Random Forest Performance Sentence Level:")
print(classification_report(y_test, y_pred_rf))

Random Forest Performance Sentence Level:
              precision    recall  f1-score   support

        -1.0       0.61      0.26      0.37       118
         1.0       0.81      0.95      0.87       382

    accuracy                           0.79       500
   macro avg       0.71      0.61      0.62       500
weighted avg       0.76      0.79      0.75       500



CountVectorizer Sentence lvl RF

In [19]:
# CountVectorizer Sentence lvl RF
# Sentence lvl RF (using CountVectorizer)

# Assuming X_train and X_test are lists of sentences
# Create a CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 1))  # Use unigrams for sentence-level representation

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Random Forest model with the vectorized data
rf_model = RandomForestClassifier(n_estimators=100, random_state=56)
rf_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_rf = rf_model.predict(X_test_vec)  # Use vectorized data for predictions

# Print the performance report
print("Random Forest Performance Sentence Level (CountVectorizer):")
print(classification_report(y_test, y_pred_rf))

Random Forest Performance Sentence Level (CountVectorizer):
              precision    recall  f1-score   support

        -1.0       0.83      0.33      0.47       118
         1.0       0.83      0.98      0.90       382

    accuracy                           0.83       500
   macro avg       0.83      0.65      0.68       500
weighted avg       0.83      0.83      0.80       500



## Logistic Regression

Word lvl LogisticRegression

In [20]:
from sklearn.linear_model import LogisticRegression # Import LogisticRegression
logreg_model = LogisticRegression()
logreg_model.fit(X_train_vec, y_train) # Use vectorized data for training
# Initialize and train the Logistic Regression model
logreg_model = LogisticRegression()
logreg_model.fit(X_train_vec, y_train) # Use vectorized data for training

# Make predictions
y_pred_logreg = logreg_model.predict(X_test_vec) # Use vectorized data for prediction

# Evaluate performance
print("Logistic Regression Performance:")
print(classification_report(y_test, y_pred_logreg))

Logistic Regression Performance:
              precision    recall  f1-score   support

        -1.0       0.60      0.46      0.52       118
         1.0       0.84      0.91      0.87       382

    accuracy                           0.80       500
   macro avg       0.72      0.68      0.70       500
weighted avg       0.79      0.80      0.79       500



Word lvl countvectorizer on logisticRegression

In [21]:
# Create a CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 1))  # Example parameters

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the Logistic Regression model
logreg_model = LogisticRegression()
logreg_model.fit(X_train_vec, y_train) # Use vectorized data for training

# Make predictions
y_pred_logreg = logreg_model.predict(X_test_vec) # Use vectorized data for prediction

# Evaluate performance
print("Logistic Regression Performance (CountVectorizer):")
print(classification_report(y_test, y_pred_logreg))

Logistic Regression Performance (CountVectorizer):
              precision    recall  f1-score   support

        -1.0       0.60      0.46      0.52       118
         1.0       0.84      0.91      0.87       382

    accuracy                           0.80       500
   macro avg       0.72      0.68      0.70       500
weighted avg       0.79      0.80      0.79       500



sentence lvl logistic regression

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Assuming X_train and X_test are lists of sentences (e.g., ["This is a sentence.", "Another sentence here."])
# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))  # Use unigrams for sentence-level representation

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)  # Each sentence is treated as a separate document

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)  # Transform test sentences into the same feature space

# Initialize and train the Logistic Regression model with the vectorized data
logreg_model = LogisticRegression(max_iter=1000)  # Increase max_iter if needed for convergence
logreg_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_logreg = logreg_model.predict(X_test_vec)  # Use vectorized data for predictions

# Evaluate performance
print("Logistic Regression Performance Sentence lvl:")
print(classification_report(y_test, y_pred_logreg))

Logistic Regression Performance Sentence lvl:
              precision    recall  f1-score   support

        -1.0       0.79      0.29      0.42       118
         1.0       0.82      0.98      0.89       382

    accuracy                           0.81       500
   macro avg       0.80      0.63      0.66       500
weighted avg       0.81      0.81      0.78       500



Countvectorizer Sentence lvl Logistic regression

In [27]:
from sklearn.feature_extraction.text import CountVectorizer  # Import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Assuming X_train and X_test are lists of sentences (e.g., ["This is a sentence.", "Another sentence here."])
# Create a CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 1))  # Use unigrams for sentence-level representation

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)  # Each sentence is treated as a separate document

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)  # Transform test sentences into the same feature space

# Initialize and train the Logistic Regression model with the vectorized data
logreg_model = LogisticRegression(max_iter=1000)  # Increase max_iter if needed for convergence
logreg_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_logreg = logreg_model.predict(X_test_vec)  # Use vectorized data for predictions

# Evaluate performance
print("Logistic Regression Performance Sentence Level:")
print(classification_report(y_test, y_pred_logreg))

Logistic Regression Performance Sentence Level:
              precision    recall  f1-score   support

        -1.0       0.60      0.46      0.52       118
         1.0       0.84      0.91      0.87       382

    accuracy                           0.80       500
   macro avg       0.72      0.68      0.70       500
weighted avg       0.79      0.80      0.79       500



Grid search on logistic regression

In [28]:

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Define the parameter grid
param_grid = {
    'C': [0.1, 1, 10],  # Regularization strength
    'penalty': ['l1', 'l2'],  # Penalty type
    'solver': ['liblinear', 'saga'] # Solver algorithm
}

# Initialize the model
logreg = LogisticRegression(max_iter=1000) # Increased max_iter for convergence

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=logreg, param_grid=param_grid, cv=5, scoring='accuracy')

# Fit the model with training data.  Convert X_train to a numerical representation
# For example, if X_train is text, use TF-IDF vectorizer.  Replace with your feature extraction method.
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
grid_search.fit(X_train_vec, y_train)


# Print the best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

# Evaluate the best model on the test set
X_test_vec = vectorizer.transform(X_test)
y_pred = grid_search.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means 

Best Parameters: {'C': 10, 'penalty': 'l2', 'solver': 'saga'}
Best Score: 0.7793333333333334
              precision    recall  f1-score   support

        -1.0       0.69      0.52      0.59       118
         1.0       0.86      0.93      0.89       382

    accuracy                           0.83       500
   macro avg       0.77      0.72      0.74       500
weighted avg       0.82      0.83      0.82       500

Accuracy: 0.83


CountVectorizer on Grid search logistic regression

In [29]:

from sklearn.feature_extraction.text import CountVectorizer


# Define the parameter grid
param_grid = {
    'C': [0.1, 1, 10],  # Regularization strength
    'penalty': ['l1', 'l2'],  # Penalty type
    'solver': ['liblinear', 'saga'] # Solver algorithm
}

# Initialize the model
logreg = LogisticRegression(max_iter=1000) # Increased max_iter for convergence

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=logreg, param_grid=param_grid, cv=5, scoring='accuracy')

# Fit the model with training data using CountVectorizer
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
grid_search.fit(X_train_vec, y_train)

# Print the best parameters and score
print("Best Parameters (CountVectorizer):", grid_search.best_params_)
print("Best Score (CountVectorizer):", grid_search.best_score_)

# Evaluate the best model on the test set
X_test_vec = vectorizer.transform(X_test)
y_pred = grid_search.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print("Accuracy (CountVectorizer):", accuracy_score(y_test, y_pred))

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

Best Parameters (CountVectorizer): {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}
Best Score (CountVectorizer): 0.7837777777777777
              precision    recall  f1-score   support

        -1.0       0.65      0.47      0.55       118
         1.0       0.85      0.92      0.88       382

    accuracy                           0.82       500
   macro avg       0.75      0.70      0.72       500
weighted avg       0.80      0.82      0.81       500

Accuracy (CountVectorizer): 0.816


## SVM

Word lvl SVM

In [30]:
from sklearn.svm import SVC

# Initialize and train the SVM model
svm_model = SVC()
svm_model.fit(X_train_vec, y_train)

# Make predictions
y_pred_svm = svm_model.predict(X_test_vec)

# Evaluate performance
print("SVM Performance word lvl:")
print(classification_report(y_test, y_pred_svm))

SVM Performance word lvl:
              precision    recall  f1-score   support

        -1.0       0.78      0.26      0.39       118
         1.0       0.81      0.98      0.89       382

    accuracy                           0.81       500
   macro avg       0.79      0.62      0.64       500
weighted avg       0.80      0.81      0.77       500



Word lvl CountVectorizer SVM

In [31]:
from sklearn.feature_extraction.text import CountVectorizer

# Create a CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 1))  # Example parameters

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Initialize and train the SVM model
svm_model = SVC()
svm_model.fit(X_train_vec, y_train)

# Make predictions
y_pred_svm = svm_model.predict(X_test_vec)

# Evaluate performance
print("SVM Performance (CountVectorizer):")
print(classification_report(y_test, y_pred_svm))

SVM Performance (CountVectorizer):
              precision    recall  f1-score   support

        -1.0       0.76      0.26      0.39       118
         1.0       0.81      0.97      0.88       382

    accuracy                           0.81       500
   macro avg       0.78      0.62      0.64       500
weighted avg       0.80      0.81      0.77       500



SVM senetnce lvl

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer  # Import TfidfVectorizer
from sklearn.svm import SVC  # Import SVC for Support Vector Classification
from sklearn.metrics import classification_report

# Assuming X_train and X_test are lists of sentences
# Create a TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))  # Use unigrams for sentence-level representation

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)  # Each sentence is treated as a separate document

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)  # Transform test sentences into the same feature space

# Initialize and train the SVM model
svm_model = SVC()  # You can also specify parameters like kernel, C, etc.
svm_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_svm = svm_model.predict(X_test_vec)  # Use vectorized data for predictions

# Evaluate performance
print("SVM Performance  (Sentence Level):")
print(classification_report(y_test, y_pred_svm))

SVM Performance  (Sentence Level):
              precision    recall  f1-score   support

        -1.0       0.84      0.26      0.40       118
         1.0       0.81      0.98      0.89       382

    accuracy                           0.81       500
   macro avg       0.82      0.62      0.64       500
weighted avg       0.82      0.81      0.77       500



using countevectorizer on Senetnce lvl SVM

In [33]:
from sklearn.feature_extraction.text import CountVectorizer  # Import CountVectorizer
from sklearn.svm import SVC  # Import SVC for Support Vector Classification
from sklearn.metrics import classification_report

# Assuming X_train and X_test are lists of sentences
# Create a CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 1))  # Use unigrams for sentence-level representation

# Fit the vectorizer to the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)  # Each sentence is treated as a separate document

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)  # Transform test sentences into the same feature space

# Initialize and train the SVM model
svm_model = SVC()  # You can also specify parameters like kernel, C, etc.
svm_model.fit(X_train_vec, y_train)  # Use vectorized data for training

# Make predictions on the vectorized test data
y_pred_svm = svm_model.predict(X_test_vec)  # Use vectorized data for predictions

# Evaluate performance
print("SVM Performance with CountVectorizer (Sentence Level):")
print(classification_report(y_test, y_pred_svm))

SVM Performance with CountVectorizer (Sentence Level):
              precision    recall  f1-score   support

        -1.0       0.76      0.26      0.39       118
         1.0       0.81      0.97      0.88       382

    accuracy                           0.81       500
   macro avg       0.78      0.62      0.64       500
weighted avg       0.80      0.81      0.77       500



#Train Embeddings Using Word2Vec

## LSTM word lvl

In [34]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer # Import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 1: Encode Labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Step 2: Compute Class Weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights)

# Initialize Tokenizer and fit on text data
tokenizer = Tokenizer() # Create a Tokenizer instance
tokenizer.fit_on_texts(X_train) # Fit the tokenizer to your training text data

# Calculate max_sequence_length
max_sequence_length = max([len(seq.split()) for seq in X_train]) # Calculate the maximum sequence length in your training data
print(f"Calculated max_sequence_length: {max_sequence_length}")


# Step 3: Recheck Embedding Alignment
vocab_size = len(tokenizer.word_index) + 1
# Define embedding dimension (e.g., 100)
embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim)) # 'embedding_dim' should be defined
for word, i in tokenizer.word_index.items():
    # 'word2vec_model' should be defined and trained before this loop
    # if word in word2vec_model.wv:
    #     embedding_matrix[i] = word2vec_model.wv[word]
    # else:
    #     embedding_matrix[i] = np.random.normal(size=(embedding_dim,))  # Handle OOV words
    embedding_matrix[i] = np.random.normal(size=(embedding_dim,))  # Initialize with random embeddings
# Pad training and test sequences to a uniform length
X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_sequence_length)
X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_sequence_length)


# Step 4: Define Improved LSTM Model
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_sequence_length,  # Now using the calculated max_sequence_length
        trainable=False  # Set to True if embeddings need fine-tuning
    ),
    Bidirectional(LSTM(128, return_sequences=False)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model before training
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy']) # Add this line

# Step 5: Train the Model with Class Weights
history = model.fit(
    X_train_padded,
    y_train_enc,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weights
)


# Step 6: Evaluate the Model
y_pred_lstm = (model.predict(X_test_padded) > 0.5).astype(int)
print("LSTM Performance with Class Weights:")
print(classification_report(y_test_enc, y_pred_lstm))
print("Accuracy:", accuracy_score(y_test_enc, y_pred_lstm))

Class Weights: {0: 1.7214996174445294, 1: 0.7046664578766051}
Calculated max_sequence_length: 49


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.5056 - loss: 0.6980 - val_accuracy: 0.6022 - val_loss: 0.6699
Epoch 2/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6502 - loss: 0.6487 - val_accuracy: 0.6467 - val_loss: 0.6514
Epoch 3/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6993 - loss: 0.5755 - val_accuracy: 0.6356 - val_loss: 0.6526
Epoch 4/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7630 - loss: 0.4907 - val_accuracy: 0.6778 - val_loss: 0.6279
Epoch 5/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8395 - loss: 0.3575 - val_accuracy: 0.6956 - val_loss: 0.6015
Epoch 6/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9106 - loss: 0.2383 - val_accuracy: 0.7044 - val_loss: 0.7674
Epoch 7/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9360 - loss: 0.1599 - val_accuracy: 0.7333 - val_loss: 0.8003
Epoch 8/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9459 - loss: 0.1262 - val_accurac

Sentence lvl LSTM


In [38]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from sklearn.metrics import classification_report
from sentence_transformers import SentenceTransformer  # Import SentenceTransformer

# Step 1: Encode Labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Step 2: Compute Class Weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights)

# Step 3: Convert sentences to embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # Load a pre-trained sentence transformer model

# Convert X_train and X_test to lists before encoding
X_train_list = X_train.tolist()  # Convert to list
X_test_list = X_test.tolist()    # Convert to list

# Encode the sentences to get embeddings using the lists
X_train_embeddings = model.encode(X_train_list, convert_to_tensor=True) #use list
X_test_embeddings = model.encode(X_test_list, convert_to_tensor=True) #use list

# Move tensors to CPU and convert to NumPy arrays
X_train_embeddings = X_train_embeddings.cpu().detach().numpy()
X_test_embeddings = X_test_embeddings.cpu().detach().numpy()

# Reshape the input data to add the timesteps dimension
X_train_embeddings = X_train_embeddings.reshape(X_train_embeddings.shape[0], 1, X_train_embeddings.shape[1])
X_test_embeddings = X_test_embeddings.reshape(X_test_embeddings.shape[0], 1, X_test_embeddings.shape[1])
# Step 4: Define Improved LSTM Model
# Note: The input shape should match the embedding size of the sentence transformer
embedding_dim = X_train_embeddings.shape[2]  # Get the embedding dimension from the reshaped embeddings
model = Sequential([
    LSTM(128, return_sequences=False, input_shape=(X_train_embeddings.shape[1], embedding_dim)),  # Adjust input shape
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model before training
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Step 5: Train the Model with Class Weights
history = model.fit(
    X_train_embeddings,
    y_train_enc,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weights
)

# Step 6: Evaluate the Model
y_pred_lstm = (model.predict(X_test_embeddings) > 0.5).astype(int)
print("LSTM Performance with Class Weights:")
print(classification_report(y_test_enc, y_pred_lstm))

Class Weights: {0: 1.7214996174445294, 1: 0.7046664578766051}
Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


127/127 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.6778 - loss: 0.6872 - val_accuracy: 0.6533 - val_loss: 0.6617
Epoch 2/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6638 - loss: 0.6521 - val_accuracy: 0.6222 - val_loss: 0.6462
Epoch 3/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6605 - loss: 0.6049 - val_accuracy: 0.6956 - val_loss: 0.5909
Epoch 4/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6698 - loss: 0.6126 - val_accuracy: 0.7089 - val_loss: 0.5652
Epoch 5/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6925 - loss: 0.5939 - val_accuracy: 0.7111 - val_loss: 0.5862
Epoch 6/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6986 - loss: 0.5791 - val_accuracy: 0.7089 - val_loss: 0.5790
Epoch 7/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6956 - loss: 0.5891 - val_accuracy: 0.6733 - val_loss: 0.6097
Epoch 8/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7009 - loss: 0.5704 - val_accuracy: 0.6867 - val_

## MLP Classifier

MLP Word lvl

In [66]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import classification_report, accuracy_score

#  X_train_vec, y_train, X_test_vec, and y_test are available
from sklearn.feature_extraction.text import TfidfVectorizer  # Or CountVectorizer

# Split dataset
X = dataset['Processed_Sentence']
y = dataset['Neutral (N) / Hostile (H)']

# Encode Labels (O -> 0, H -> 1), handling NaN values
y = y.map({'O': 0, 'H': 1}).fillna(-1).astype(int)  # Handle NaN and ensure integer type

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=56)

# Create and fit the vectorizer on the training data ONLY
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))  # Or CountVectorizer
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)


# Initialize the MLP classifier
mlp = MLPClassifier(max_iter=500)  # Reduce max_iter to speed up tuning

# Define a smaller parameter grid
param_distributions = {
    'hidden_layer_sizes': [(50,), (100,)],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001]
}

# Use RandomizedSearchCV for faster tuning
random_search_mlp = RandomizedSearchCV(
    mlp,
    param_distributions=param_distributions,
    n_iter=5,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)
random_search_mlp.fit(X_train_vec, y_train)

# Print the best hyperparameters and score
print("Best Hyperparameters:", random_search_mlp.best_params_)
print("Best Score:", random_search_mlp.best_score_)

# Evaluate the best model on the test set
y_pred_mlp = random_search_mlp.predict(X_test_vec)
print(classification_report(y_test, y_pred_mlp))
print("Accuracy:", accuracy_score(y_test, y_pred_mlp))


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Hyperparameters: {'solver': 'adam', 'hidden_layer_sizes': (100,), 'alpha': 0.0001, 'activation': 'relu'}
Best Score: 0.762249970288674
              precision    recall  f1-score   support

          -1       0.54      0.49      0.52       251
           1       0.84      0.86      0.85       749

    accuracy                           0.77      1000
   macro avg       0.69      0.68      0.68      1000
weighted avg       0.76      0.77      0.76      1000

Accuracy: 0.767


MLP Sentence lvl

In [47]:
pip install sentence-transformers

In [67]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
from sentence_transformers import SentenceTransformer

# Instead of using the example data, use your original dataset
X = dataset['Sentence']  # Use the 'Sentence' column from your DataFrame
y = dataset['Neutral (N) / Hostile (H)']  # Use the 'Label' column

# Encode Labels (O -> 0, H -> 1)
y = y.map({'O': 0, 'H': 1}).fillna(-1).astype(int)  # Handle NaN and ensure integer type

# Split dataset with a larger training set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=56) # Increased test_size for more training data

# Step 1: Generate Sentence Embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # Load a pre-trained sentence transformer model

# Encode the sentences to get embeddings
X_train_vec = model.encode(X_train.tolist()) # Convert to list before encoding
X_test_vec = model.encode(X_test.tolist())   # Convert to list before encoding

# Step 2: Initialize the MLP classifier
mlp = MLPClassifier(max_iter=500)  # Reduce max_iter to speed up tuning

# Step 3: Define a smaller parameter grid
param_distributions = {
    'hidden_layer_sizes': [(50,), (100,)],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001]
}

# Step 4: Use RandomizedSearchCV for faster tuning
random_search_mlp = RandomizedSearchCV(
    mlp,
    param_distributions=param_distributions,
    n_iter=5,
    cv=3, # or cv=5
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

# Step 5: Fit the model
random_search_mlp.fit(X_train_vec, y_train)

# Print the best hyperparameters and score
print("Best Hyperparameters:", random_search_mlp.best_params_)
print("Best Score:", random_search_mlp.best_score_)

# Step 6: Evaluate the best model on the test set
y_pred_mlp = random_search_mlp.predict(X_test_vec)
print(classification_report(y_test, y_pred_mlp))
print("Accuracy:", accuracy_score(y_test, y_pred_mlp))

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Hyperparameters: {'solver': 'adam', 'hidden_layer_sizes': (100,), 'alpha': 0.001, 'activation': 'relu'}
Best Score: 0.7272472166017517
              precision    recall  f1-score   support

          -1       0.47      0.48      0.48       251
           1       0.83      0.82      0.82       749

    accuracy                           0.74      1000
   macro avg       0.65      0.65      0.65      1000
weighted avg       0.74      0.74      0.74      1000

Accuracy: 0.736
